# Qwen NL2SH API on Kaggle + ngrok

Serve **kovidritesh/qwen2.5-coder-3b-nl2sh-alfa-qlora** on top of
**Qwen/Qwen2.5-Coder-3B-Instruct**. This notebook loads the actual rank-64 adapter
with PEFT and exposes an authenticated, OpenAI-style, non-streaming
`POST /v1/chat/completions` endpoint through ngrok.

## Before running

1. Upload this `.ipynb` into a **private Kaggle Notebook**.
2. In **Settings / Session options**, enable **Internet** and select a GPU
   accelerator, such as **GPU T4 x2** or **P100**, if available. Only GPU 0 is used.
   Enable GPU access/account verification if Kaggle requests it.
3. Create an ngrok account and copy its **authtoken** from
   [the ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).
4. In **Add-ons → Secrets**, add and grant this notebook access to:
   - `NGROK_AUTHTOKEN`: your ngrok authtoken.
   - `NL2SH_API_KEY`: a private random API key of **at least 32 characters**.
     Generate one locally with `python -c "import secrets; print(secrets.token_urlsafe(32))"`.
   Secret prompts are available as a fallback; do not paste credentials into code cells.
5. Run the cells in order in an **interactive session**. The last cell stays
   running to serve requests. Interrupt that cell to shut down.

This is a temporary development endpoint, subject to Kaggle session/GPU quotas
and ngrok account limits. It stops when the session stops. The final cell does
not bypass Kaggle's session limits. Keep the notebook private and do not publish
credentials in outputs. No generated command is executed by this notebook.


In [ ]:
# Keep Kaggle's preinstalled CUDA-enabled torch; install only serving/model helpers.
%pip install -q "transformers>=4.51,<5" "peft>=0.15,<0.19" "accelerate>=1.6,<2" "fastapi>=0.115,<1" "uvicorn>=0.34,<1" "pyngrok==9.1.4" "requests>=2.32,<3"


If installation asks for a kernel restart, restart the session/kernel and resume
at the next cell. Model imports must happen after dependency installation.
The public model repositories do not require a Hugging Face token.


In [ ]:
import getpass
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle Settings and restart this session.")


def read_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        value = getpass.getpass(f"Enter {name} (hidden; or configure Kaggle Secrets): ")
    value = value.strip()
    if not value:
        raise ValueError(f"{name} must not be empty.")
    return value


NGROK_AUTHTOKEN = read_secret("NGROK_AUTHTOKEN")
API_KEY = read_secret("NL2SH_API_KEY")
if len(API_KEY) < 32 or not API_KEY.isascii() or any(c.isspace() for c in API_KEY):
    raise ValueError("NL2SH_API_KEY must be at least 32 ASCII characters with no whitespace.")

DEVICE = "cuda:0"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| dtype:", DTYPE)
print("Secrets loaded; their values are hidden.")


In [ ]:
from peft import PeftConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"
BASE_REVISION = "488639f1ff808d1d3d0ba301aef8c11461451ec5"
ADAPTER_MODEL = "kovidritesh/qwen2.5-coder-3b-nl2sh-alfa-qlora"
ADAPTER_REVISION = "2c851234f1c731fbbdbf848d9e080238653a1bb7"
MODEL_NAME = "nl2sh"
MAX_CONTEXT_TOKENS = 4096

adapter_config = PeftConfig.from_pretrained(ADAPTER_MODEL, revision=ADAPTER_REVISION)
if adapter_config.base_model_name_or_path != BASE_MODEL:
    raise ValueError("Adapter configuration does not match the selected base model.")

# Use the tokenizer saved with the fine-tuned adapter, including its chat template.
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_MODEL, revision=ADAPTER_REVISION)
if not tokenizer.chat_template:
    raise ValueError("The adapter tokenizer has no chat template.")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# FP16/BF16 weights fit on one 16 GB GPU; no bitsandbytes installation is needed.
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    revision=BASE_REVISION,
    torch_dtype=DTYPE,
    device_map={"": DEVICE},
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_MODEL,
    revision=ADAPTER_REVISION,
    is_trainable=False,
)
model.eval()
print(f"Loaded {MODEL_NAME}: {BASE_MODEL} + LoRA rank {adapter_config.r}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 2**30:.2f} GiB")


In [ ]:
import secrets
import threading
import time
import uuid
from typing import Annotated, Literal

from fastapi import Depends, FastAPI, HTTPException
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from pydantic import BaseModel, ConfigDict, Field, FiniteFloat

bearer = HTTPBearer(auto_error=False)


def authorize(credentials: Annotated[HTTPAuthorizationCredentials | None, Depends(bearer)]):
    if credentials is None or not secrets.compare_digest(
        credentials.credentials.encode(), API_KEY.encode()
    ):
        raise HTTPException(401, "Invalid or missing API key", headers={"WWW-Authenticate": "Bearer"})


class Message(BaseModel):
    model_config = ConfigDict(extra="forbid")
    role: Literal["system", "user", "assistant"]
    content: str = Field(min_length=1, max_length=16000)


class ChatRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    model: Literal["nl2sh"] = "nl2sh"
    messages: list[Message] = Field(min_length=1, max_length=32)
    max_tokens: int = Field(default=256, ge=1, le=512)
    temperature: FiniteFloat = Field(default=0.0, ge=0, le=2)
    stream: bool = False


app = FastAPI(title="Qwen NL2SH", docs_url=None, redoc_url=None, openapi_url=None)
generation_lock = threading.Lock()


@app.get("/health", dependencies=[Depends(authorize)])
def health():
    return {"status": "ok", "model": MODEL_NAME, "adapter": ADAPTER_MODEL}


@app.get("/v1/models", dependencies=[Depends(authorize)])
def list_models():
    return {"object": "list", "data": [{"id": MODEL_NAME, "object": "model", "owned_by": "local"}]}


@app.post("/v1/chat/completions", dependencies=[Depends(authorize)])
def chat(body: ChatRequest):
    if body.stream:
        raise HTTPException(400, "This endpoint supports stream=false only.")
    if sum(len(message.content) for message in body.messages) > 24000:
        raise HTTPException(413, "Combined message text exceeds 24000 characters.")
    if not generation_lock.acquire(blocking=False):
        raise HTTPException(429, "GPU is busy; retry after the current request completes.", headers={"Retry-After": "5"})
    try:
        prompt = tokenizer.apply_chat_template(
            [message.model_dump() for message in body.messages],
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
        input_tokens = inputs["input_ids"].shape[1]
        if input_tokens + body.max_tokens > MAX_CONTEXT_TOKENS:
            raise HTTPException(400, f"Input + max_tokens must be <= {MAX_CONTEXT_TOKENS} tokens.")
        inputs = inputs.to(DEVICE)
        generation_args = {
            "max_new_tokens": body.max_tokens,
            "do_sample": body.temperature > 0,
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
            "use_cache": True,
        }
        if body.temperature > 0:
            generation_args["temperature"] = body.temperature
        with torch.inference_mode():
            output = model.generate(**inputs, **generation_args)
        new_ids = output[0, input_tokens:].tolist()
        text = tokenizer.decode(new_ids, skip_special_tokens=True)
        finish_reason = "stop" if new_ids and new_ids[-1] == tokenizer.eos_token_id else "length"
        return {
            "id": "chatcmpl-" + uuid.uuid4().hex,
            "object": "chat.completion",
            "created": int(time.time()),
            "model": MODEL_NAME,
            "choices": [{"index": 0, "message": {"role": "assistant", "content": text}, "finish_reason": finish_reason}],
            "usage": {"prompt_tokens": input_tokens, "completion_tokens": len(new_ids), "total_tokens": input_tokens + len(new_ids)},
        }
    finally:
        generation_lock.release()


print("API configured. Generation is limited to one request at a time.")


In [ ]:
import requests
import uvicorn

if "server_thread" in globals() and server_thread.is_alive():
    raise RuntimeError("Server is already running. Stop it with the shutdown instructions before rerunning this cell.")

PORT = 8000
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning", access_log=False))
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

deadline = time.monotonic() + 30
while not server.started:
    if not server_thread.is_alive() or time.monotonic() > deadline:
        server.should_exit = True
        raise RuntimeError("API failed to start. Check the server output and whether port 8000 is already in use.")
    time.sleep(0.1)

HEADERS = {"Authorization": f"Bearer {API_KEY}", "ngrok-skip-browser-warning": "1"}
LOCAL_URL = f"http://127.0.0.1:{PORT}"
assert requests.get(LOCAL_URL + "/health", timeout=10).status_code == 401
response = requests.get(LOCAL_URL + "/health", headers=HEADERS, timeout=10)
response.raise_for_status()
print("Local API ready; authentication checks passed.")


In [ ]:
# Check inference locally before exposing the model to the internet.
TEST_PAYLOAD = {
    "model": "nl2sh",
    "messages": [
        {"role": "system", "content": "Translate the request into a shell command. Return only the command."},
        {"role": "user", "content": "List all files including hidden files in the current directory."},
    ],
    "temperature": 0,
    "max_tokens": 128,
}
invalid = {**TEST_PAYLOAD, "max_tokens": 0}
assert requests.post(LOCAL_URL + "/v1/chat/completions", headers=HEADERS, json=invalid, timeout=10).status_code == 422
response = requests.post(LOCAL_URL + "/v1/chat/completions", headers=HEADERS, json=TEST_PAYLOAD, timeout=180)
response.raise_for_status()
completion = response.json()
assert completion["model"] == MODEL_NAME
assert completion["choices"][0]["message"]["content"].strip(), "Model returned an empty completion."
print("Model response:", completion["choices"][0]["message"]["content"])
print("PASS: local authentication, request validation, and model generation.")


In [ ]:
from pyngrok import conf, ngrok

if "public_tunnel" in globals():
    raise RuntimeError("A tunnel was already created. Use the shutdown instructions before rerunning this cell.")

ngrok_config = conf.PyngrokConfig(auth_token=NGROK_AUTHTOKEN, config_version="2")
public_tunnel = ngrok.connect(addr=PORT, proto="http", bind_tls=True, pyngrok_config=ngrok_config)
PUBLIC_URL = public_tunnel.public_url.rstrip("/")

try:
    unauthenticated = requests.get(PUBLIC_URL + "/health", headers={"ngrok-skip-browser-warning": "1"}, timeout=30)
    assert unauthenticated.status_code == 401, f"Unexpected unauthenticated response: {unauthenticated.status_code}"
    response = requests.get(PUBLIC_URL + "/health", headers=HEADERS, timeout=30)
    response.raise_for_status()
    assert response.json()["model"] == MODEL_NAME
except Exception:
    ngrok.disconnect(PUBLIC_URL, pyngrok_config=ngrok_config)
    del public_tunnel
    raise

print("Public base URL:", PUBLIC_URL + "/v1")
print("Chat endpoint:", PUBLIC_URL + "/v1/chat/completions")
print("Use your NL2SH_API_KEY as the Bearer token. Its value is not printed.")


## Call the endpoint from your own machine

Copy the printed URL above. Use the value you saved as `NL2SH_API_KEY`; the ngrok
authtoken is only for opening the tunnel and is **not** the inference API key.

```bash
export NL2SH_URL='https://YOUR-PRINTED-NGROK-URL'
read -rsp 'NL2SH API key: ' NL2SH_API_KEY; echo
export NL2SH_API_KEY
curl --fail-with-body "$NL2SH_URL/v1/chat/completions" \
  -H "Authorization: Bearer $NL2SH_API_KEY" \
  -H 'ngrok-skip-browser-warning: 1' \
  -H 'Content-Type: application/json' \
  -d '{"model":"nl2sh","messages":[{"role":"system","content":"Translate the request into a shell command. Return only the command."},{"role":"user","content":"Find Python files in the current directory."}],"max_tokens":128,"temperature":0}'
```

With the optional `openai` client **on your machine** (`pip install openai`):

```python
import os
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["NL2SH_URL"].rstrip("/") + "/v1",
    api_key=os.environ["NL2SH_API_KEY"],
    default_headers={"ngrok-skip-browser-warning": "1"},
    timeout=180,
)
result = client.chat.completions.create(
    model="nl2sh",
    messages=[{"role": "user", "content": "Show disk usage of this directory."}],
    max_tokens=128,
    temperature=0,
)
print(result.choices[0].message.content)
```

Supported: plain-text system/user/assistant messages, `model="nl2sh"`,
`max_tokens` (1–512), `temperature` (0–2), `stream=false`.
Input + requested output is limited to 4,096 tokens. Concurrent generation
returns HTTP 429; retry when the GPU is free. Streaming and tool calls are not
implemented. Match your training-time prompt format for best results; the model
card does not document it. The notebook uses the adapter's saved chat template.

## Keep serving / stop

Run the next cell to keep the notebook executing while you use the API.
Interrupt it to disconnect this tunnel and stop this server. Kaggle's normal
session limits still apply. After stopping, restart from the server cell; the
model can stay loaded in GPU memory.

If you need to stop before running the final cell, run this in a new code cell:

```python
if "public_tunnel" in globals():
    ngrok.disconnect(public_tunnel.public_url, pyngrok_config=ngrok_config)
    del public_tunnel
if "server" in globals():
    server.should_exit = True
if "server_thread" in globals():
    server_thread.join(timeout=15)
```


In [ ]:
print("Serving at", PUBLIC_URL, "— interrupt this cell to stop.")
try:
    while server_thread.is_alive():
        time.sleep(5)
finally:
    try:
        if "public_tunnel" in globals():
            ngrok.disconnect(public_tunnel.public_url, pyngrok_config=ngrok_config)
            del public_tunnel
    finally:
        server.should_exit = True
        server_thread.join(timeout=15)
        print("Tunnel disconnected and server stop requested.")


## Troubleshooting and references

- **No GPU / CUDA out of memory:** enable a GPU session; restart the session to
  unload any other models. This notebook uses only one GPU and half-precision
  weights; reduce `MAX_CONTEXT_TOKENS` and request `max_tokens` if necessary.
- **Import errors after installation:** restart the kernel and run from the
  configuration cell. Do not import Transformers before the install cell.
- **Downloads fail:** enable Internet and retry; check Hugging Face availability.
- **ngrok error:** verify the authtoken, account setup, and active-tunnel limits.
  If your account requires a specific endpoint domain, add `domain="YOUR-DOMAIN"`
  to `ngrok.connect(...)` using the domain assigned in your ngrok dashboard.
- **401:** use `NL2SH_API_KEY` as `Authorization: Bearer ...`.
- **API disappears:** check whether Kaggle stopped the session or ngrok reached
  an account limit. Rerun the notebook in an available GPU session.

Sources: [Kaggle Notebooks](https://www.kaggle.com/docs/notebooks),
[PEFT model loading](https://huggingface.co/docs/peft/package_reference/peft_model),
[pyngrok API](https://pyngrok.readthedocs.io/en/stable/api.html),
[adapter repository](https://huggingface.co/kovidritesh/qwen2.5-coder-3b-nl2sh-alfa-qlora).
